# Import libraries

In [5]:
import pandas as pd
import numpy as np
import requests
import statsmodels.formula.api as smf
# from stargazer.stargazer import Stargazer
import json
apikey = 'IbVrD08MpXh0yJhfYXGe9eFNlHwRWkXNyDb5RWws'
import tqdm
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import datetime
import altair as alt

# Data extract

In [ ]:
print('Started csv load @ ' + str(datetime.datetime.now()))

wash19incidents = pd.read_csv('washington/NIBRS_incident.csv')
wash19incidents['INCIDENT_MONTH'] = pd.DatetimeIndex(wash19incidents['INCIDENT_DATE']).month
wash19arrestees = pd.read_csv('washington/NIBRS_ARRESTEE.csv')
wash19offenseTypes = pd.read_csv('washington/NIBRS_OFFENSE_TYPE.csv')
relevant_offenses = ['Trespass of Real Property','Curfew/Loitering/Vagrancy Violations','Drug/Narcotic Violations','Drug Equipment Violations','Drunkenness','Disorderly Conduct','Family Offenses, Nonviolent']
wash19agencies = pd.read_csv('washington/agencies.csv')
wash19race = pd.read_csv('washington/REF_RACE.csv')
wash19justForce = pd.read_csv('washington/NIBRS_JUSTIFIABLE_FORCE.csv')
wash19victim = pd.read_csv('washington/NIBRS_VICTIM.csv')
wash19victimCirc = pd.read_csv('washington/NIBRS_VICTIM_CIRCUMSTANCES.csv')
wash19victimInj= pd.read_csv('washington/NIBRS_VICTIM_INJURY.csv')
wash19victimOffense = pd.read_csv('washington/NIBRS_VICTIM_OFFENSE.csv')
wash19victimType = pd.read_csv('washington/NIBRS_VICTIM_TYPE.csv')
wash19circCodes = pd.read_csv('washington/NIBRS_CIRCUMSTANCES.csv')
wash19injCodes = pd.read_csv('washington/NIBRS_INJURY.csv')

wash18incidents = pd.read_csv('washington/2018/NIBRS_incident.csv')
wash18incidents['INCIDENT_MONTH'] = pd.DatetimeIndex(wash18incidents['INCIDENT_DATE']).month
wash18arrestees = pd.read_csv('washington/2018/NIBRS_ARRESTEE.csv')
wash18offenseTypes = pd.read_csv('washington/2018/NIBRS_OFFENSE_TYPE.csv')
wash18agencies = pd.read_csv('washington/2018/agencies.csv')
wash18race = pd.read_csv('washington/2018/REF_RACE.csv')
wash18victim = pd.read_csv('washington/2018/NIBRS_VICTIM.csv')
wash18victimCirc = pd.read_csv('washington/2018/NIBRS_VICTIM_CIRCUMSTANCES.csv')
wash18victimInj= pd.read_csv('washington/2018/NIBRS_VICTIM_INJURY.csv')
wash18victimOffense = pd.read_csv('washington/2018/NIBRS_VICTIM_OFFENSE.csv')
wash18victimType = pd.read_csv('washington/2018/NIBRS_VICTIM_TYPE.csv')
wash18circCodes = pd.read_csv('washington/2018/NIBRS_CIRCUMSTANCES.csv')
wash19injCodes = pd.read_csv('washington/NIBRS_INJURY.csv')

print('Finished csv load @ ' + str(datetime.datetime.now()))

Started csv load @ 2021-08-25 19:20:14.920090


In [ ]:
display(wash18victim)
display(wash19victimCirc)
display(wash18circCodes)
display(wash19victimInj)
display(wash19victimType)

In [ ]:
print('Started merge @ ' + str(datetime.datetime.now()))

wash19offenses = pd.read_csv('washington/NIBRS_OFFENSE.csv')
wash19offenses = wash19offenses.merge(wash19incidents, how='left', on='INCIDENT_ID')
wash19offenses = wash19offenses.merge(wash19agencies, how='left', on='AGENCY_ID')
wash19offenses = wash19offenses.merge(wash19offenseTypes, how='left', on='OFFENSE_TYPE_ID')
wash19offenses = wash19offenses.merge(wash19arrestees, how='left', on='INCIDENT_ID')
wash19offenses = wash19offenses.merge(wash19race, how='left', on='RACE_ID')
wash19location = pd.read_csv('washington/NIBRS_LOCATION_TYPE.csv')
wash19offenses = wash19offenses.merge(wash19location, how='left', on='LOCATION_ID')

wash18offenses = pd.read_csv('washington/2018/NIBRS_OFFENSE.csv')
wash18offenses = wash18offenses.merge(wash18incidents, how='left', on='INCIDENT_ID')
wash18offenses = wash18offenses.merge(wash18agencies, how='left', on='AGENCY_ID')
wash18offenses = wash18offenses.merge(wash18offenseTypes, how='left', on='OFFENSE_TYPE_ID')
wash18offenses = wash18offenses.merge(wash18arrestees, how='left', on='INCIDENT_ID')
wash18offenses = wash18offenses.merge(wash18race, how='left', on='RACE_ID')
# wash18offenses = wash18offenses.merge(wash18race, how='left', left_on='RACE_ID_x',right_on='RACE_ID')
wash18location = pd.read_csv('washington/2018/NIBRS_LOCATION_TYPE.csv')
wash18offenses = wash18offenses.merge(wash18location, how='left', on='LOCATION_ID')

print('Finished merge @ ' + str(datetime.datetime.now()))

In [ ]:
#need to figure out which columns were added to arrestees dataframe and drop those from 18/19
wash1819offenses = pd.concat([wash19offenses,wash18offenses])
# wash1819offenses

In [ ]:
olympia19offenses = wash19offenses[wash19offenses['COUNTY_NAME']=='THURSTON']
olympia19offenseCounts = pd.DataFrame(olympia19offenses.groupby(['INCIDENT_MONTH','OFFENSE_NAME'])['OFFENSE_NAME'].count())
pd.set_option('display.max_rows', 500)
olympia19offenseCounts = olympia19offenseCounts.rename(columns={'OFFENSE_NAME':'OFFENSE_COUNT'})
olympia19offenseCounts = olympia19offenseCounts.reset_index()
olympia19offenseCounts['CRU'] = np.where(olympia19offenseCounts['INCIDENT_MONTH']<4,0,1)
relevant_offenses = ['Trespass of Real Property','Curfew/Loitering/Vagrancy Violations','Drug/Narcotic Violations','Drug Equipment Violations','Drunkenness','Disorderly Conduct','Family Offenses, Nonviolent']
olympia19offenseCounts = olympia19offenseCounts[olympia19offenseCounts['OFFENSE_NAME'].isin(relevant_offenses)]
# olympia19offenseCounts = olympia19offenseCounts[olympia19offenseCounts['OFFENSE_NAME']=='Drug/Narcotic Violations']
# olympia19offenseCounts['Rolling'] = olympia19offenseCounts['OFFENSE_COUNT'].rolling(8).mean()

olympia1819offenses = wash1819offenses[wash1819offenses['COUNTY_NAME']=='THURSTON']
olympia1819offenses['INCIDENT_YEAR'] = pd.DatetimeIndex(olympia1819offenses['INCIDENT_DATE']).year
olympia1819offenseCounts = pd.DataFrame(olympia1819offenses.groupby(['INCIDENT_YEAR','INCIDENT_MONTH','OFFENSE_NAME'])['OFFENSE_NAME'].count())
pd.set_option('display.max_rows', 500)
olympia1819offenseCounts = olympia1819offenseCounts.rename(columns={'OFFENSE_NAME':'OFFENSE_COUNT'})
olympia1819offenseCounts = olympia1819offenseCounts.reset_index()
olympia1819offenseCounts['CRU'] = np.where((olympia1819offenseCounts['INCIDENT_MONTH']<4)|(olympia1819offenseCounts['INCIDENT_YEAR']==2018),0,1)
relevant_offenses = ['Trespass of Real Property','Curfew/Loitering/Vagrancy Violations','Drug/Narcotic Violations','Drug Equipment Violations','Drunkenness','Disorderly Conduct','Family Offenses, Nonviolent']
# olympia1819offenseCounts = olympia1819offenseCounts[olympia1819offenseCounts['OFFENSE_NAME'].isin(relevant_offenses)]
olympia1819offenseCounts = olympia1819offenseCounts[olympia1819offenseCounts['OFFENSE_NAME']=='Drug/Narcotic Violations']
# olympia19offenseCounts['Rolling'] = olympia19offenseCounts['OFFENSE_COUNT'].rolling(8).mean()

In [ ]:
olympia19chart = alt.Chart(olympia19offenseCounts).mark_line().encode(
    x='INCIDENT_MONTH:O',
    y='OFFENSE_COUNT',
    color='OFFENSE_NAME'
)

In [ ]:
olympia1819chart = alt.Chart(olympia1819offenseCounts,width=800, height=200).mark_line().encode(
    x='INCIDENT_MONTH:O',
    y='OFFENSE_COUNT',
    color=alt.Color('INCIDENT_YEAR:O',scale=alt.Scale(range=['#D15B41','#4B3D3D']))
)
display(olympia1819chart)
# olympia1819offenseCounts

In [ ]:
#NEXT STEPS 8-18-2021#
#MERGE IN VICTIM DATA
#CAN LOOK AT CIT FOR OLYMPIA / STAR FOR DENVER

# Supervised models

## KNN classification

### Model development

### Visualization

### Analysis

## Logistic regression

### Model development

### Visualization

### Analysis

## Linear regression

### Model development

### Visualization

### Analysis

## OLS regression

### Model development

In [ ]:
### Example code ####

#`smf.ols(formula="y ~ treatment + x1 + x2 ...")#
#"fixed effect for var_x", you can add a `+C(var_x)`

#OFFENSE OLS REGRESSION#
########################
covariates = ['Intercept','CRU','RACE_DESC','LOCATION_NAME','water_2006','apr_may_07']
modelA = smf.ols(formula='OFFENSE_COUNT ~ CRU', data=a2).fit()
modelB = smf.ols(formula='OFFENSE_COUNT ~ CRU + RACE_DESC +C(route)', data=a2).fit()
modelC = smf.ols(formula='OFFENSE_COUNT ~ CRU + RACE_DESC + LOCATION_NAME +C(route)', data=a2).fit()
Q3_table = Stargazer([modelA,modelB,modelC])
Q3_table.covariate_order(covariates)
Q3_table
#######################
#######################

#VICTIM OLS REGRESSION##
########################
covariates = ['Intercept','CRU','RACE_DESC','LOCATION_NAME','water_2006','apr_may_07']
modelA = smf.ols(formula='VICTIM_COUNT ~ CRU', data=a2).fit()
modelB = smf.ols(formula='VICTIM_COUNT ~ CRU + RACE_DESC +C(route)', data=a2).fit()
modelC = smf.ols(formula='VICTIM_COUNT ~ CRU + RACE_DESC + LOCATION_NAME +C(route)', data=a2).fit()
Q3_table = Stargazer([modelA,modelB,modelC])
Q3_table.covariate_order(covariates)
Q3_table
########################
########################

### Visualization

### Analysis

# Unsupervised models

## PCA

### Model development

### Visualization

### Analysis

## DBSCAN

### Model development

### Visualization

### Analysis

## API

In [ ]:
### API call library ###
########################

# https://crime-data-explorer.fr.cloud.gov/pages/docApi

########################
########################


In [ ]:
url = "https://api.usa.gov/crime/fbi/sapi/api/agencies/list?api_key=" + apikey
headers = {'Accept': 'application/json'}

response = requests.get(url, headers=headers)

data = json.loads(response.content)
agencies = pd.DataFrame(data)
oris = list(agencies['ori'].unique())

In [ ]:
#summarized-data-controller#
#second one - Agency level SRS crime data endpoint#
offenses = []
# for ori in tqdm(oris[:100]):
for ori in oris[:100]:
    try:
        url = f"https://api.usa.gov/crime/fbi/sapi/api/summarized/agencies/{ori}/offenses/2019/2021?api_key=" + apikey
        
        headers = {'Accept': 'application/json'}

        response = requests.get(url, headers=headers)

        data = json.loads(response.content)
        offenses.append(pd.DataFrame(data['results']))
    except:
        pass
    
offensesDf = pd.concat(offenses)
display(offensesDf)

In [ ]:
url = f"https://api.usa.gov/crime/fbi/sapi/api/summarized/agencies/CODPD0000/offenses/2016/2018?api_key=" + apikey

headers = {'Accept': 'application/json'}

response = requests.get(url, headers=headers)

data = json.loads(response.content)
# pd.DataFrame(data['results'])

In [ ]:
#arrest-tkm-controller
offensesDemo = []
KC = 'MOKPD0000'
for ori in oris[:100]:
    try:
        url = f"https://api.usa.gov/crime/fbi/sapi/api/arrest/agencies/offense/{ori}/all/2018/2020?api_key=" + apikey
#         url = f"https://api.usa.gov/crime/fbi/sapi/api/arrest/agencies/offense/{KC}/drug/2019/2021?api_key=" + apikey
        headers = {'Accept': 'application/json'}

        response = requests.get(url, headers=headers)

        data = json.loads(response.content)
        display(data)
#         offensesDemo.append(pd.DataFrame(data['results']))
    except:
        pass
    
# offensesDemoDf = pd.concat(offensesDemo)

# url = f"https://api.usa.gov/crime/fbi/sapi/api/arrest/agencies/offense/{KC}/drug/2019/2021?api_key=" + apikey
# headers = {'Accept': 'application/json'}

# response = requests.get(url, headers=headers)

# data = json.loads(response.content)
# display(data)

# url = f"https://api.usa.gov/crime/fbi/sapi/api/arrest/agencies/{KC}/drug-grand-total/race/2019/2021?api_key=" + apikey
# headers = {'Accept': 'application/json'}

# response = requests.get(url, headers=headers)

# data = json.loads(response.content)
# display(data)

In [ ]:
agencies = requests.get('https://api.usa.gov/crime/fbi/sapi/api/nibrs/property-crime',auth=('key',apikey))
data = json.loads(agencies.content)
# pd.DataFrame(data)
data